In [1]:
##### MODEL DETAILS ---------------------------------------------------------------------

# Training name
training_name = 'MaxConv2D_HQ_40e_2bit'

# Initial thresholds from transformer
initial_thresholds = [120,510,1384]

# Gaussian noise parameters
NOISE_MU = 0.0
NOISE_SIGMA = 40.0 # e-

# Precision of input data
N_BITS = 2

In [2]:
# IMPORTS ---------------------------------------------------------------------

import warnings
warnings.filterwarnings("ignore")

import os
import random

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

from DG.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import custom_loss
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler

from models import *

2026-06-04 11:17:29.728126: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
pi = 3.14159265359
maxval=1e9
minval=1e-9

In [4]:
# TRAINING DATA ---------------------------------------------------------------------

dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/"
tfrecords_base_dir = "/uscms/home/jennetd/nobackup/smart-pixels/tfrecords"

dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'train_contained/')
dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3sr_16x16_'+str(int(NOISE_SIGMA))+'eNoise_train')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'3sr_16x16_'+str(int(NOISE_SIGMA))+'eNoise_test')


In [5]:
print("Running training of model " + training_name)   
with open('log_'+training_name+'.txt','a') as f:
    f.write("Running training of model " + training_name + "\n")
    
seed = random.randint(0, 1000)
print("Seed: ", seed)
with open('log_'+training_name+'.txt','a') as f:
    f.write('Seed: ' + str(seed) + "\n")
    
# Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=seed,
    quantize=False,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=seed,
    quantize=False,
)
  

Running training of model MaxConv2D_HQ_40e_2bit
Seed:  904
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_40eNoise_test/metadata.json


Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_40eNoise_train/metadata.json


In [6]:
print("Initial thresholds: ", initial_thresholds)
with open('log_'+training_name+'.txt','a') as f:
    f.write("Initial thresholds: " + str(initial_thresholds) + "\n")

model = CreateHQModel(shape = (16,16,2), 
                      output = 14, 
                      n_filters=5,
                      pool_size=3,
                      thresholds=initial_thresholds)

model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3, clipnorm=1.0),
    loss=custom_loss,
)

fingerprint = '%08x' % random.randrange(16**8)

print("Fingerprint: ", fingerprint)
with open('log_'+training_name+'.txt','a') as f:
    f.write('Fingerprint: ' + str(fingerprint) + "\n")
    
base_dir = f'/uscms/home/jennetd/nobackup/smart-pixels/noise-paper/trained_models/model-{fingerprint}-{training_name}-checkpoints'
checkpoints_dir = os.path.join(base_dir, 'checkpoints')
checkpoint_filepath = os.path.join(checkpoints_dir, 'weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5')
    
os.makedirs(base_dir, exist_ok=True)
os.makedirs(checkpoints_dir, exist_ok=True) 

Initial thresholds:  [120, 510, 1384]
Fingerprint:  8a298200


In [7]:
early_stopping_patience = 500
es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    save_freq='epoch'
)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)

In [ ]:
history = model.fit(
    x=training_generator,
    validation_data=validation_generator,
    callbacks=[es,mcp,csv_logger],
    epochs=5000,
    shuffle=True,
    verbose=1
)

Epoch 1/5000
84/84 [==============================] - 10s 86ms/step - loss: 48120.7812 - val_loss: 8128.5308
Epoch 2/5000
84/84 [==============================] - 7s 78ms/step - loss: 4437.1304 - val_loss: 1338.9604
Epoch 3/5000
84/84 [==============================] - 7s 83ms/step - loss: 3558.8608 - val_loss: 192.4964
Epoch 4/5000
84/84 [==============================] - 6s 73ms/step - loss: -281.2976 - val_loss: -1150.5920
Epoch 5/5000
84/84 [==============================] - 6s 73ms/step - loss: -788.7211 - val_loss: -1471.6226
Epoch 6/5000
84/84 [==============================] - 6s 71ms/step - loss: -750.1062 - val_loss: -600.1753
Epoch 7/5000
84/84 [==============================] - 6s 71ms/step - loss: -1741.7529 - val_loss: 2162.4529
Epoch 8/5000
84/84 [==============================] - 6s 70ms/step - loss: -1630.7964 - val_loss: -2518.3218
Epoch 9/5000
84/84 [==============================] - 6s 69ms/step - loss: -2117.9041 - val_loss: -2310.4944
Epoch 10/5000
84/84 [========